# SleepFM — embeddings NSRR pour la confirmation de l'IAH (GPU)

Fichier : `notebooks/sleepfm_nsrr_embeddings.ipynb`. Plan validé le 13 novembre 2026
(`docs/plans/Plan confirmation IAH NSRR SleepFM OSF 13-11.md`).

Embeddings SleepFM et OSF des nuits de la **liste figée** (`configs/nsrr_<cohorte>_subjects.json`,
commitée le 13 novembre à partir de l'index public NSRR : 2 056 nuits MESA, 402 Go d'EDF), avec la règle d'échelle `physionet_median_v1`. Les EDF ne tiennent pas
sur le disque : ils sont téléchargés par lots depuis NSRR, traités puis supprimés. Aucun label,
aucun score : l'IAH se calcule ensuite sur le Mac, une seule fois (`scripts/evaluate_nsrr_ahi.py`).

**À vérifier avant le premier run** : l'accord d'utilisation NSRR doit autoriser le calcul sur
Kaggle (aucune donnée n'est republiée, les EDF sont supprimés après lecture).

**Sur Kaggle** :
1. `Settings > Accelerator > GPU T4 x2` et `Settings > Internet > On`.
2. `Add-ons > Secrets` : `NSRR_TOKEN` (jeton personnel sleepdata.org).
3. Essai court : `LIMIT = 2`, puis `Run All` ; vérifier les libellés EDF et le téléchargement.
4. Run complet : `LIMIT = None`, puis `Save Version > Save & Run All (Commit)`.
5. Télécharger depuis `Output` : `embeddings_nsrr_<cohorte>.zip`, `nsrr_log.txt`, `pip_freeze.txt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Settings > Accelerator"
!df -h . | tail -1

## 1. Code, dépendances et poids (empreintes de la règle figée)

In [ ]:
import os, hashlib
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
REVISION = "origin/main"  # ou le SHA exact du commit, noté dans les sorties
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
OSF_REVISION = "f2067f2d95b3bff43e4a690a471ba3d4457f12a7"
!mkdir -p third_party/osf
![ -f third_party/osf/osf_backbone.pth ] || curl -sL -o third_party/osf/osf_backbone.pth https://huggingface.co/yang-ai-lab/OSF-Base/resolve/{OSF_REVISION}/osf_backbone.pth
SLEEPFM_CKPT = "third_party/sleepfm/sleepfm/checkpoint/best.pt"
OSF_CKPT = "third_party/osf/osf_backbone.pth"
EXPECTED = {  # mêmes poids que sur PhysioNet (results/respiratory_ahi_physionet2018.json)
    SLEEPFM_CKPT: "a2fe7e5abf70f8a448c5577a085e0b7096332dcb1ec7497916e425446ac24575",
    OSF_CKPT: "c51190b1942556969af3c3d63c2e59430ddb1ea0377c50ea87df83712fc31857",
}
for path, sha in EXPECTED.items():
    digest = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if digest != sha:
        raise RuntimeError(f"{path} : empreinte inattendue {digest}")
print("Poids SleepFM et OSF vérifiés")

## 2. Paramètres

- `COHORT` : `mesa` d'abord, `shhs1` ensuite si l'accès est accordé.
- `LIMIT` : `2` pour l'essai court, `None` pour toute la liste figée.
- `LOT` : nombre d'EDF téléchargés avant chaque passage (limité par le disque, environ 20 Go).

In [ ]:
COHORT = "mesa"
LIMIT = 2  # None pour le run complet
LOT = 10
NSRR_DATASET = {"mesa": "mesa", "shhs1": "shhs"}[COHORT]
NSRR_EDF_PATH = {"mesa": "polysomnography/edfs", "shhs1": "polysomnography/edfs/shhs1"}[COHORT]  # chemins vérifiés sur l'index public NSRR le 13 nov.
SUBJECTS = f"configs/nsrr_{COHORT}_subjects.json"
OUT = f"{WORK}/embeddings/nsrr_{COHORT}"
EDF_DIR = f"{WORK}/edf_lot"
import json
subjects = json.load(open(SUBJECTS))["subjects"]  # erreur ici : liste pas encore figée et commitée
print(f"{len(subjects)} nuit(s) dans la liste figée {SUBJECTS}")

## 3. Téléchargement NSRR (gem officielle)

Le jeton est lu dans les secrets Kaggle et n'apparaît dans aucune sortie.

In [ ]:
!apt-get -qq install -y ruby-full > /dev/null && gem install nsrr --no-document -q > /dev/null && nsrr version
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("NSRR_TOKEN")
import re, subprocess, shutil
from pathlib import Path

def download(lot):
    """Télécharge les EDF du lot dans EDF_DIR (gem nsrr : un dossier filtré par nom, MD5 vérifié).

    Renvoie les sujets non obtenus. La gem recrée <dataset>/<dossier>/ sous le dossier courant.
    """
    pattern = "^(" + "|".join(re.escape(s) for s in lot) + r")\.edf$"
    result = subprocess.run(
        ["nsrr", "download", f"{NSRR_DATASET}/{NSRR_EDF_PATH}", "--shallow", f"--file={pattern}", f"--token={TOKEN}"],
        cwd=EDF_DIR, capture_output=True, text=True,
    )
    folder = Path(EDF_DIR) / NSRR_DATASET / NSRR_EDF_PATH
    missing = []
    for subject in lot:
        local = folder / f"{subject}.edf"
        if local.exists():
            local.rename(Path(EDF_DIR) / f"{subject}.edf")
        else:
            missing.append(subject)
    if missing:
        print(f"Non téléchargés : {missing} ; fin de la sortie nsrr : {result.stdout[-300:].replace(TOKEN, '***')}")
    return missing

## 4. Embeddings par lots

Chaque lot : téléchargement, `scripts/embed_nsrr.py` (reprend les nuits déjà faites), suppression
des EDF. Un EDF introuvable reste « manquant » et le run est alors déclaré incomplet.

In [ ]:
def pending():
    return [s for s in subjects if not (Path(OUT) / f"{s}.npz").exists() and not (Path(OUT) / f"{s}.skipped").exists()]

command = [
    "python", "-u", "scripts/embed_nsrr.py", "--cohort", COHORT, "--edf-dir", EDF_DIR,
    "--subjects", SUBJECTS, "--out", OUT, "--checkpoint", SLEEPFM_CKPT,
    "--respiratory-rule", "physionet_median_v1", "--osf-checkpoint", OSF_CKPT, "--batch-size", "128",
]
todo = pending()[:LIMIT] if LIMIT else pending()
unavailable = []
with open(f"{WORK}/nsrr_log.txt", "a") as log:
    for start in range(0, len(todo), LOT):
        shutil.rmtree(EDF_DIR, ignore_errors=True)
        os.makedirs(EDF_DIR)
        lot = todo[start : start + LOT]
        unavailable += download(lot)
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            log.write(line)
            if "EDF absent" not in line:
                print(line, end="")
        process.wait()  # code 1 tant que la liste n'est pas finie : normal entre deux lots
        print(f"--- lot {start // LOT + 1} : {len(subjects) - len(pending())}/{len(subjects)} nuits traitées")
shutil.rmtree(EDF_DIR, ignore_errors=True)
print(f"Introuvables sur NSRR : {unavailable}")

## 5. Garder le résultat

Sur le Mac, dézipper dans `embeddings/nsrr_<cohorte>/`, puis lancer l'évaluation une seule fois.

In [ ]:
status = json.load(open(f"{OUT}/run_status.json"))
print(f"{len(status['done'])} nuit(s) faites, {len(status['rejected'])} rejetée(s), manquantes : {status['missing'][:5]}")
if not status["complete"]:
    print("Run incomplet : relancer la section 4 pour reprendre (l'évaluation le refusera sinon)")
!python -m pip freeze > {WORK}/pip_freeze.txt
shutil.make_archive(f"{WORK}/embeddings_nsrr_{COHORT}", "zip", OUT)
!ls -lh {WORK}/*.zip {WORK}/*.txt